# OER Study Objects: End-to-End to OH Adsorption Values

This notebook demonstrates the object workflow from `OER_catalyst_study` to `OER_surface_study` and `OER_adsorption_site_study`, ending with OH adsorption/free-energy values.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from ase.io import read

# Ensure local package import works when running from notebooks/
repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from gnome_dreams_oer_screening import OER_catalyst_study


In [2]:
# Load a test bulk structure
bulk_atoms = read(repo_root / "notebooks" / "TiO2_rutile.cif")
bulk_atoms

Atoms(symbols='Ti2O4', pbc=True, cell=[4.59983732, 4.59983732, 2.95921356], spacegroup_kinds=...)

In [3]:
# Initialize catalyst study and rank likely surfaces
study = OER_catalyst_study(
    bulk_atoms.copy(),
    H2O_gas_free_energy=-14.217,
    H2_gas_free_energy=-6.77,
    use_init_as_relaxed_bulk=True,
)

study.predict_most_likely_surfaces(max_miller=1, method="O-coordination")
ranking = study.get_termination_rankings().sort_values(
    "Normalized score", ascending=False
)
display(ranking[["Termination index", "Miller indices", "Normalized score"]].head())


,Termination index,Miller indices,Normalized score
2,2,"(1, 1, 0)",1.000000
4,4,"(1, 0, 0)",0.593465
3,3,"(1, 0, 1)",0.479541
5,5,"(0, 0, 1)",0.195581
1,1,"(1, 1, 1)",0.000000


In [4]:
# Initialize one surface study and set relaxed surface
termination_index = int(ranking.iloc[0]["Termination index"])
study.initialize_oer_surface_study(termination_index)
surface_study = study.get_surface_studies()[termination_index]

surface_atoms = surface_study.get_non_relaxed_surface()
surface_ok = surface_study.set_relaxed_base_surface(
    surface_atoms,
    energy=-500.0,
)
print("set_relaxed_base_surface ->", surface_ok)

surface_study.determine_adsorption_sites()
sites_df = surface_study.get_adsorption_sites_df()
display(sites_df[["Site index", "site type", "ad site element"]].head())


set_relaxed_base_surface -> True


,Site index,site type,ad site element
0,0,on-top,Ti
1,1,on-top,Ti
2,2,lattice O,N/A
3,3,lattice O,N/A
4,4,lattice O,N/A


In [5]:
# Pick a site and initialize adsorption-site study
if (sites_df["site type"] == "on-top").any():
    site_index = int(
        sites_df[sites_df["site type"] == "on-top"]["Site index"].iloc[0]
    )
else:
    site_index = int(sites_df["Site index"].iloc[0])

surface_study.initialize_adsorption_site_study(site_index)
ad_study = surface_study.get_adsorption_site_studies_dict()[site_index]
print("Chosen site index:", site_index, "site type:", ad_study.site_type)


Chosen site index: 0 site type: on-top


In [6]:
# Set relaxed O calculation result (new API returns success/message)
atoms_for_O = ad_study.get_atoms_for_O_calc()
ok_O, msg_O = ad_study.set_relaxed_O_calc(atoms_for_O, energy=-498.75)
print("set_relaxed_O_calc ->", ok_O, msg_O)
if not ok_O:
    raise RuntimeError(msg_O)


set_relaxed_O_calc -> True None


In [7]:
# Build OH candidates and set relaxed OH results
OH_candidates = ad_study.get_OH_adsorbed_atoms()
oh_energies = [-499.40, -499.20, -499.30]
ok_OH, msg_OH = ad_study.set_relaxed_OH_calc(
    OH_candidates,
    energies=oh_energies,
)
print("set_relaxed_OH_calc ->", ok_OH, msg_OH)
if not ok_OH:
    raise RuntimeError(msg_OH)

print("Stored adsorption-site energies:")
print(ad_study.get_adsorption_site_energies())


set_relaxed_OH_calc -> True None
Stored adsorption-site energies:
{'clean_surface_energy': -500.0, 'O_adsorbed_energy': -498.75, 'OH_adsorbed_energy': -499.4, 'OOH_adsorbed_energy': None}


In [8]:
# Update derived adsorption free energies and inspect OH-related outputs
surface_study.update_adsorption_energies()
final_df = surface_study.get_adsorption_sites_df()

preferred_cols = [
    "Site index",
    "site type",
    "G_approx(O)",
    "G_approx(OH)",
    "ideal overpotential",
]
cols = [c for c in preferred_cols if c in final_df.columns]
display(final_df.loc[[site_index], cols])


,Site index,site type,G_approx(O),G_approx(OH),ideal overpotential
0,0,on-top,8.767,11.793,10.56
